# 07 | Probability and Sampling

This notebook uses a BornMachine for full probability, marginal probability, continuous inverse-CDF sampling, and discrete projector sampling.

A randomly initialized model is enough to demonstrate the API. Its samples are not expected to have meaningful statistical structure until the model has been trained.


In [1]:
from pathlib import Path
import sys

# This works whether Jupyter starts in the repository root or in notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "tneq_qc").is_dir() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/yuch3n/Documents/Code/Github/tneq-qc


In [2]:
import numpy as np

from tneq_qc import (
    BackendFactory,
    BornMachine,
    DataGenerator,
    DiscreteDataGenerator,
    EngineCommon,
    QCTNHelper,
)

backend = BackendFactory.create_backend("pytorch", device="cpu", dtype="float32")
engine = EngineCommon(backend=backend, strategy="row_priority")

graph = QCTNHelper.mps(2, bond_dim=2, phys_dim=2)
model = BornMachine(graph, 2, backend=backend).auto_init(orthogonal=True)
combined = model.build()
mx_names = model.mx_core_names
print("Measurement cores:", mx_names)


Measurement cores: ['mx.a', 'mx.b']


## 1. Full probability

`full_probability()` requires a tensor for every measurement core. The engine temporarily replaces the cores, contracts the model, and restores the original values afterward.


In [3]:
continuous_gen = DataGenerator(backend, mx_K=2)
point = np.array([[0.25, -0.50]], dtype=np.float32)
point_mx, _ = continuous_gen.generate(point, K=2, ret_type="TNTensor")

full_mx = dict(zip(mx_names, point_mx))
p_full = engine.full_probability(combined, full_mx)
print("Full density/probability value:", p_full)


[Compiler] Strategy candidates: ['row_priority'], Testing 1 strategies...
  [row_priority] Compatibility: True
  [row_priority] Estimated cost: 5.00e+05 FLOPs
[Compiler] Selected strategy: row_priority (cost: 5.00e+05)
Full density/probability value: 0.08726706355810165


## 2. Marginal probability

`marginal_probability()` accepts only a subset of Mx cores. Unspecified measurements temporarily become identities, which traces out those dimensions.


In [4]:
p_marginal = engine.marginal_probability(
    combined,
    {mx_names[0]: point_mx[0]},
)
print("Marginal density for the first dimension:", p_marginal)


Marginal density for the first dimension: 0.19911862909793854


## 3. Continuous sampling

`sample()` evaluates density on a grid and applies inverse-CDF sampling one dimension at a time. Cost grows approximately with `dimensions × samples × grid_size`, so start small.

The default `use_marginal=False` performs autoregressive conditional sampling. With `use_marginal=True`, each dimension is sampled independently from its marginal distribution.


In [5]:
samples = engine.sample(
    combined,
    continuous_gen,
    sample_core_names=mx_names,
    num_samples=20,
    bounds=(-2.0, 2.0),
    grid_size=100,
)

print("Sample shape:", samples.shape)
print(samples[:5])


Sampling:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling: 100%|██████████| 2/2 [00:00<00:00, 664.87it/s]

Sample shape: torch.Size([20, 2])
tensor([[ 0.6488, -0.1642],
        [-1.4586,  1.2279],
        [ 1.5279, -1.1748],
        [ 1.3804, -1.1010],
        [-1.8752, -1.4347]])


## 4. Discrete projectors and sampling

`DiscreteDataGenerator` maps categorical values to diagonal projectors. The basis dimension `K` must be divisible by the number of values so each category receives an equal subspace.


In [6]:
discrete_gen = DiscreteDataGenerator(
    backend,
    values=(0, 1),
    mx_K=2,
)
print(discrete_gen.projector_table())


[[[1. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 1.]]]


In [7]:
discrete_samples = engine.sample_discrete(
    combined,
    discrete_gen,
    sample_core_names=mx_names,
    num_samples=20,
    values=(0, 1),
)

print(discrete_samples.shape)
print(discrete_samples[:10])


Discrete sampling:   0%|          | 0/2 [00:00<?, ?it/s]

Discrete sampling: 100%|██████████| 2/2 [00:00<00:00, 481.69it/s]

torch.Size([20, 2])
tensor([[1., 1.],
        [0., 0.],
        [1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.],
        [0., 0.]])


## 5. Interpreting probability outputs correctly

- A continuous model returns density; a pointwise density may exceed 1.
- Discrete candidates are normalized before drawing a category.
- A random model demonstrates mechanics, not a learned distribution.
- Feature mapping, integration bounds, and normalization all affect probability semantics.
- Tiny negative outputs can come from numerical error. Sampling clamps them to zero, but persistent negatives should be investigated.

### Exercises

- Plot one marginal density on a grid over `[-2,2]`.
- Compare autoregressive samples with independent marginal samples.
- Use categories `(0,1,2)` and choose a `K` divisible by 3.
